# GLiNER2 기본 사용법

본 노트북은 GLiNER2.5 multi 체크포인트로 한국어 문장에서 엔티티, 분류, 구조화 레코드, 관계를 추출하는 방법을 다룬다. 개념은 [개념과 동작 구조](00_gliner2_overview.ipynb)에 정리했다.

모든 셀은 CPU에서 로컬로 실행되며 외부 API를 호출하지 않는다. 체크포인트는 첫 실행 시 Hugging Face에서 내려받는다(약 1.2GB).

## 준비

### 경고 출력 설정

체크포인트를 불러올 때 출력되는 경고를 숨긴다. 경고의 원인을 처리하지는 않는다.

| 경고 | 출처 | 의미 |
|---|---|---|
| `Redirects are currently not supported in MacOs` | PyTorch 분산 실행 모듈 | macOS에서 출력 리디렉션을 지원하지 않는다는 안내. 이 노트북과 무관하다. |
| `You are sending unauthenticated requests to the HF Hub` | Hugging Face Hub | 토큰 없이 내려받으면 속도 제한이 낮다는 안내. 공개 체크포인트라 토큰 없이도 받을 수 있다. |
| `You are using a model of type gliner2 to instantiate a model of type extractor` | Transformers | GLiNER2 체크포인트를 `gliner2` 패키지의 클래스로 불러올 때 나오는 형식 불일치 안내. 패키지가 의도한 동작이다. |

Transformers와 Hugging Face Hub는 처음 import될 때 자체 로그 수준을 설정하므로, `logging.getLogger(...).setLevel()`이 아니라 각 라이브러리의 `set_verbosity_error()`로 수준을 낮춘다.

In [ ]:
import logging
import warnings

warnings.filterwarnings("ignore", category=UserWarning)
logging.getLogger("torch.distributed.elastic").setLevel(logging.ERROR)

from huggingface_hub.utils import logging as hf_logging
from transformers.utils import logging as transformers_logging

hf_logging.set_verbosity_error()
transformers_logging.set_verbosity_error()

### 형태소 분할기

한국어 형태소 분할기는 프로젝트의 [`config/extractors.py`](../config/extractors.py)에서 가져온다. 프로젝트 루트가 Python 모듈 검색 경로에 포함되어야 한다.

In [ ]:
from config.extractors import kiwi_splitter

### 모델 불러오기

`AutoExtractor.from_pretrained()`는 체크포인트의 아키텍처를 확인하여 GLiNER2.5 체크포인트를 `BoundaryExtractor`로 불러온다.
이후 모든 셀은 같은 `model`을 사용한다.

In [ ]:
from gliner2 import AutoExtractor

model = AutoExtractor.from_pretrained("fastino/gliner2.5-multi-v1")
print(f"✓ {type(model).__name__} loaded")

## 엔티티 추출

### 단어 분할기와 결과

`extract_entities(text, labels)`에 레이블 이름 목록을 넘긴다. 같은 문장을 기본 분할기(`whitespace`)와 형태소 분할기(`kiwi_splitter`)로 각각 추출하여 비교한다.

In [ ]:
TEXT = "JeekLee는 2023년부터 무신사에서 백엔드 개발자로 일하고 있고, 지금은 성수동에 산다."
LABELS = ["person", "organization", "location"]

for name, splitter in (("whitespace", "whitespace"), ("kiwi", kiwi_splitter)):
    model.set_word_splitter(splitter)
    print(f"{name:>10}: {model.extract_entities(TEXT, LABELS)['entities']}")

| 분할기 | 결과 |
|---|---|
| `whitespace` | person: `JeekLee는`, organization: `무신사에서`, location: `성수동에` |
| `kiwi` | person: `JeekLee`, organization: `무신사`, location: `성수동` |

공백 분할에서는 `JeekLee는`, `무신사에서`, `성수동에`처럼 조사가 붙은 단어 전체가 하나의 후보이므로, 조사를 뺀 범위는 아예 고를 수 없다. 타입은 맞게 골랐지만 이름에 조사가 붙은 채로 저장된다. 형태소 분할에서는 조사가 별도 토큰이 되어 정확한 범위가 나온다. 로마자 이름 뒤에 한국어 조사가 붙은 `JeekLee는`도 `JeekLee` + `는`으로 나뉜다. 이후 셀은 형태소 분할기를 사용한다.

### 설명, confidence, 원문 위치

레이블을 `{"이름": "설명"}`으로 주면 설명이 레이블 임베딩에 반영된다. `include_confidence=True`는 점수를, `include_spans=True`는 원문의 문자 위치(`start`, `end`)를 함께 반환한다. 위치는 그래프 저장 시 원문과 엔티티를 연결하거나, 같은 이름이 여러 번 나올 때 구분하는 데 쓴다.

In [ ]:
LABELS_WITH_DESCRIPTION = {
    "person": "사람 이름",
    "organization": "회사나 기관 이름",
    "location": "지역이나 장소 이름",
}
result = model.extract_entities(TEXT, LABELS_WITH_DESCRIPTION, include_confidence=True, include_spans=True)
for label, items in result["entities"].items():
    for item in items:
        print(f"{label:>12}: {item['text']!r} [{item['start']}:{item['end']}] {item['confidence']:.2f}")

| 레이블 | 텍스트 | 위치 | confidence |
|---|---|---|---|
| person | JeekLee | 0:7 | 1.00 |
| organization | 무신사 | 17:20 | 0.97 |
| location | 성수동 | 44:47 | 1.00 |

`start`/`end`는 원문 문자열의 위치이며 `TEXT[17:20] == "무신사"`이다.

### 레이블 집합의 영향

각 범위는 주어진 레이블 중에서 타입을 고른다. 직책이 회사명 바로 뒤에 붙은 문장으로, 알맞은 레이블이 없을 때 생기는 오분류가 레이블을 추가하면 어떻게 바뀌는지 확인한다.

In [ ]:
ROLE_TEXT = "JeekLee는 무신사 플랫폼 엔지니어로 일하고 있다."
for labels in (LABELS, LABELS + ["job title", "date"]):
    print(labels)
    print("   ", model.extract_entities(ROLE_TEXT, labels, include_confidence=True)["entities"])

| 레이블 집합 | `무신사 플랫폼 엔지니어` | confidence |
|---|---|---|
| person, organization, location | organization ✗ | 0.95 |
| + job title, date | **job title** | 0.72 |

알맞은 레이블이 없으면 범위는 주어진 레이블 중 가장 가까운 것으로 분류되고, 그 confidence가 높을 수도 있다(0.95). `job title`을 추가하면 같은 범위가 job title로 옮겨 간다. 레이블과 문장을 함께 인코딩하므로, 레이블 집합이 바뀌면 기존 레이블의 점수도 달라진다. 따라서 추출하지 않을 타입이라도 헷갈리기 쉬운 레이블(`job title` 등)을 함께 주면 원하는 타입의 오분류가 줄어든다. 추가한 레이블의 결과도 함께 반환되므로 필요 없는 타입은 후처리에서 버린다.

다만 범위 자체는 여전히 `무신사`를 포함한다. 레이블 추가는 타입을 바로잡을 뿐이며, 이 문장에서 `무신사`를 organization으로 따로 뽑지는 못했다. 레이블이 판정의 기준이지만, 범위의 경계까지 해결해 주지는 않는다.

### Threshold

threshold를 낮게(0.05) 두고 한 번 추출한 뒤, 각 기준에서 남는 엔티티를 비교한다. 같은 결과를 기준마다 다시 추출하는 것과 같다.

In [ ]:
SENTENCE = "JeekLee 팀장은 다음 주 부산에서 열리는 AWS 서밋에 카카오 개발자들과 함께 참석한다."
labels = ["person", "organization", "location", "event", "job title", "date"]
candidates = model.extract_entities(SENTENCE, labels, threshold=0.05, include_confidence=True)["entities"]
scored = sorted(((item["confidence"], label, item["text"]) for label, items in candidates.items() for item in items),
                reverse=True)
for confidence, label, text in scored:
    kept = " ".join("✓" if confidence >= th else "·" for th in (0.2, 0.5, 0.8))
    print(f"{confidence:.2f}  [{kept}]  {text} ({label})")
print("columns: threshold 0.2 / 0.5 / 0.8")

| confidence | 엔티티 | 0.2 | 0.5 | 0.8 |
|---|---|---|---|---|
| 1.00 | 부산 (location) | ✓ | ✓ | ✓ |
| 0.98 | AWS 서밋 (event) | ✓ | ✓ | ✓ |
| 0.92 | JeekLee (person) | ✓ | ✓ | ✓ |
| 0.88 | 카카오 (organization) | ✓ | ✓ | ✓ |
| 0.86 | 팀장 (job title) | ✓ | ✓ | ✓ |
| 0.81 | AWS (organization) | ✓ | ✓ | ✓ |
| 0.70 | 다음 주 (date) | ✓ | ✓ | · |
| 0.16 | 카카오 개발자 (job title) | · | · | · |
| 0.10 | 카카오 개발자 (person) | · | · | · |

- 기본값 0.5에서는 명확한 엔티티 7개가 남고, 0.8로 올리면 `다음 주`가 빠진다.
- `AWS 서밋`(event) 안의 `AWS`(organization)처럼, 레이블이 다르면 겹치는 범위가 함께 나올 수 있다.
- 0.2 미만의 후보 `카카오 개발자`는 정답 기준에 따라 판단이 갈린다. 개체명(NER) 기준에서 person은 고유한 이름을 뜻하므로 "카카오의 개발자들"이라는 일반 명사구는 person이 아니고, 직책도 `개발자`만이 해당하며 `카카오`는 organization으로 따로 본다. 반면 대화 기억처럼 "누구와 함께 갔는가"가 중요한 용도라면 이 표현도 사람 집단을 가리키는 유효한 언급으로 볼 수 있다. 어느 쪽이든 원하는 결과는 레이블 정의(설명)로 모델에 알려 주어야 하며, threshold는 이 판단을 대신하지 못한다.
- 이 문장에서는 낮은 점수의 후보가 걸러졌지만, 모델이 확신하는 오류는 기준을 올려도 남는다. 또한 `개발자`가 job title로 나오지 않은 이유는 점수가 아니라 범위의 경계 문제이며, 아래에서 설명으로 해결한다. 위 "레이블 집합의 영향"에서 `무신사 플랫폼 엔지니어`가 organization으로 0.95를 받은 것이 그 예이다.

### 범위의 경계와 레이블 설명

위 결과에서 `개발자`는 job title로 나오지 않았다. 같은 레이블 안에서 범위가 겹치면 점수가 높은 하나만 남는데, 모델이 이 부분의 직책 범위로 `카카오 개발자`를 골랐기 때문이다. threshold는 이미 정해진 후보 중 무엇을 남길지만 고르므로, 범위가 어디서 시작하고 끝나는지는 바꾸지 못한다.

같은 문장에서 job title 레이블의 조건만 바꿔 결과를 비교한다.

In [ ]:
CONDITIONS = {
    "job title만": ["job title"],
    "레이블 6개": labels,
    "job title에 설명 추가": {
        "person": "사람 이름",
        "organization": "회사나 기관 이름",
        "location": "지역이나 장소 이름",
        "event": "행사 이름",
        "job title": "직업이나 직책 (예: 개발자, 팀장, 디자이너)",
        "date": "날짜나 기간",
    },
}
for name, condition in CONDITIONS.items():
    result = model.extract_entities(SENTENCE, condition, threshold=0.1, include_confidence=True)["entities"]
    found = ", ".join(f"{item['text']} {item['confidence']:.2f}" for item in result["job title"])
    print(f"{name:>16}: {found}")

| 조건 | job title 결과 |
|---|---|
| `job title`만 | `JeekLee 팀장` 0.87, `카카오 개발자` 0.60 |
| 레이블 6개 (이름만) | `팀장` 0.86, `카카오 개발자` 0.16 |
| 레이블 6개 + 설명 | **`팀장` 0.95, `개발자` 0.90** |

- **레이블이 하나뿐이면** 이름과 회사까지 직책 범위에 들어간다(`JeekLee 팀장`, `카카오 개발자`).
- **person, organization을 함께 주면** `JeekLee`와 `카카오`를 다른 레이블이 가져가 `팀장`은 정리되지만, `카카오 개발자`는 여전히 한 덩어리이다.
- **설명을 붙이면** `개발자`, `팀장`이 정확한 경계와 높은 점수로 나온다. 설명에 든 예시(`개발자, 팀장, 디자이너`)가 "직책은 소속을 뺀 직업명"이라는 기준을 모델에 전달한다.

**레이블 설명은 선택 사항이 아니라 추출 기준을 정하는 수단이다.** GLiNER는 레이블과 문장을 함께 인코딩해 판정하므로, 레이블 이름만 주면 모델이 학습 때 익힌 막연한 의미로 범위를 고른다. 설명과 예시는 무엇을 하나의 엔티티로 볼지(경계), 어떤 타입으로 볼지(분류)를 직접 정한다. 앞의 "레이블 집합의 영향"에서 `무신사 플랫폼 엔지니어`가 organization으로 잡힌 것처럼, 타입 오류와 경계 오류는 대부분 threshold가 아니라 레이블 집합과 설명으로 고친다. 실제 적용에서는 레이블마다 짧은 정의와 예시를 붙이고, 대표 문장 몇 개로 결과를 확인하면서 설명을 다듬는 것이 좋다.

### 여러 문장 처리

`batch_extract_entities()`는 문장 목록을 묶어서 처리한다. 결과는 입력 순서대로 반환된다.

In [ ]:
batch = [
    TEXT,
    "네이버클라우드와 KT가 공동으로 판교에 데이터센터를 짓기로 했습니다.",
    "JeekLee 팀장은 다음 주 부산에서 열리는 AWS 서밋에 카카오 개발자들과 함께 참석한다.",
]
for sentence, result in zip(batch, model.batch_extract_entities(batch, LABELS)):
    print(sentence)
    print("   ", result["entities"])

세 문장 모두 사람·조직·장소가 조사 없이 추출된다. 세 번째 문장에서는 `AWS 서밋`의 `AWS`가 organization으로 나온다. 이 레이블 집합에는 event가 없어서, 행사 이름의 일부가 조직으로 분류되었다.

## 분류

`classify_text(text, tasks)`는 작업 이름과 후보 레이블을 받아 문장 전체에 레이블을 붙인다. 엔티티 추출과 같은 모델과 스키마 체계를 사용한다.

In [ ]:
reviews = ["배송이 너무 늦고 포장도 엉망이었어요.", "생각보다 화면이 밝고 배터리도 오래 가네요."]
for review in reviews:
    print(review, model.classify_text(review, {"sentiment": ["positive", "negative", "neutral"]},
                                      include_confidence=True))

| 문장 | 결과 | 판정 |
|---|---|---|
| 배송이 너무 늦고 포장도 엉망이었어요. | negative 0.999 | ✓ |
| 생각보다 화면이 밝고 배터리도 오래 가네요. | negative **0.499** | ✗ |

두 번째 문장은 긍정 리뷰지만 negative로 분류되었고, confidence도 0.5 근처로 판단이 불확실하다. multi 체크포인트의 분류 헤드는 한국어 문장 분류에서 신뢰하기 어렵다. 공식 문서는 분류에 특화된 `fastino/GLiNER2.5-multi-Decide` 체크포인트를 따로 제공한다.

## 구조화 레코드

`extract_json(text, structures)`는 레코드 이름과 필드 목록을 받아 필드를 채운 레코드를 반환한다. 필드는 `"이름::타입::설명"` 형식으로 쓴다. 숫자와 단위가 붙은 값(`256GB`, `169만`)에서 두 분할기의 결과를 비교한다.

In [27]:
PRODUCT = "갤럭시 S25 울트라 256GB 모델은 169만 8400원에 출시됐다."
STRUCTURE = {"product": ["name::str::제품명", "storage::str::저장 용량", "price::str::가격"]}
for name, splitter in (("whitespace", "whitespace"), ("kiwi", kiwi_splitter)):
    model.set_word_splitter(splitter)
    print(f"{name:>10}: {model.extract_json(PRODUCT, STRUCTURE)}")
model.set_word_splitter(kiwi_splitter)  # 이후 셀은 형태소 분할기를 사용한다
print("✓ splitter: kiwi")

whitespace: {'product': [{'name': '갤럭시 S25 울트라 256GB', 'storage': '256GB', 'price': '169만 8400원에'}]}
      kiwi: {'product': [{'name': '갤럭시 S25 울트라', 'storage': '256', 'price': '169'}, {'name': '갤럭시 S25 울트라 256GB', 'storage': '256GB', 'price': '169만 8400원'}]}
✓ splitter: kiwi


| 분할기 | 결과 |
|---|---|
| `whitespace` | 레코드 1개: name `갤럭시 S25 울트라 256GB`, storage `256GB`, price **`169만 8400원에`** |
| `kiwi` | 레코드 2개: ① name `갤럭시 S25 울트라`, storage **`256`**, price **`169`** ② name `갤럭시 S25 울트라 256GB`, storage `256GB`, price `169만 8400원` |

두 분할기가 서로 다른 방식으로 틀린다. 공백 분할은 가격에 조사(`에`)가 붙는다. 형태소 분할은 `256GB`를 `256`과 `GB`로, `169만`을 `169`와 `만`으로 나누므로 숫자만 있는 짧은 후보가 생기고, 이를 채운 불완전한 레코드가 하나 더 나온다. 숫자와 단위가 붙은 값을 다룰 때는 형태소 분할기가 숫자·단위를 한 토큰으로 유지하도록 조정하거나, 레코드를 후처리에서 병합해야 한다.

## 관계

### 독립 관계 추출

`extract_relations(text, relations)`는 관계 타입마다 (머리, 꼬리) 범위 쌍을 반환한다. 머리·꼬리가 어떤 엔티티 타입이어야 하는지는 지정하지 않는다.

In [28]:
print(model.extract_relations(TEXT, ["works_for", "lives_in"], include_confidence=True))

{'relation_extraction': {'works_for': [{'head': {'text': 'JeekLee', 'confidence': 0.9985130429267883}, 'tail': {'text': '무신사', 'confidence': 0.9674007892608643}, 'confidence': 0.7455814480781555}], 'lives_in': [{'head': {'text': 'JeekLee', 'confidence': 0.9643149971961975}, 'tail': {'text': '성수동', 'confidence': 0.9884241223335266}, 'confidence': 0.7575797438621521}]}}


`works_for`는 `JeekLee → 무신사`(관계 confidence 0.75), `lives_in`은 `JeekLee → 성수동`(0.76)으로 추출되었다. 관계 이름만 주었지만 머리·꼬리 범위의 confidence는 0.96 이상이다. 다만 머리·꼬리의 엔티티 타입은 검사하지 않으므로, 문장에 따라 타입이 맞지 않는 쌍이 나올 수 있다.

### 스키마 결합

`create_schema()`로 엔티티, 관계, 분류를 하나의 스키마에 선언하고 `extract()`로 한 번에 추출한다. 관계에도 설명을 붙일 수 있다.

In [29]:
schema = (
    model.create_schema()
    .entities(LABELS_WITH_DESCRIPTION)
    .relations({"works_for": "사람이 조직에서 일한다", "lives_in": "사람이 장소에 산다"})
    .classification("topic", ["career", "travel", "finance"])
)
print(model.extract(TEXT, schema))

{'entities': {'person': ['JeekLee'], 'organization': ['무신사'], 'location': ['성수동']}, 'topic': 'career', 'relation_extraction': {'works_for': [('JeekLee', '무신사')], 'lives_in': [('JeekLee', '성수동')]}}


```python
{'entities': {'person': ['JeekLee'], 'organization': ['무신사'], 'location': ['성수동']},
 'topic': 'career',
 'relation_extraction': {'works_for': [('JeekLee', '무신사')], 'lives_in': [('JeekLee', '성수동')]}}
```

엔티티, 관계, 분류 결과가 한 번의 호출로 함께 나온다. 분류 결과는 `career`이다. 결합 스키마에서도 관계의 머리·꼬리 타입은 강제되지 않으며, 타입 제약이 필요하면 아래 JointIE를 쓴다.

### JointIE

JointIE는 Joint Information Extraction(엔티티·관계 공동 정보 추출)의 약자이다. `JointIEEngine`은 같은 모델 가중치를 공유하면서 엔티티와 관계를 함께 디코딩한다. 관계마다 머리·꼬리 엔티티 타입과 그래프 제약(`unique_head` 등)을 선언한다. 결과의 관계는 엔티티 ID(`e1`, `e2`, …)로 끝점을 가리킨다.

In [30]:
from gliner2.joint_ie import JointIEConfig, JointIEEngine

joint = JointIEEngine(model, device="cpu")
joint_schema = (
    joint.create_schema()
    .entity("person", "사람 이름")
    .entity("organization", "회사나 기관 이름")
    .entity("location", "지역이나 장소 이름")
    .relation("works_for", ["person"], ["organization"], "사람이 조직에서 일한다", unique_head=True)
    .relation("lives_in", ["person"], ["location"], "사람이 장소에 산다")
)
result = joint.extract(TEXT, joint_schema, config=JointIEConfig(entity_threshold=0.2))
names = {entity.id: entity.text for entity in result.entities}
print("feasible:", result.feasible)
for entity in result.entities:
    print(f"  {entity.id} {entity.text} ({entity.type}) {entity.confidence:.2f}")
for relation in result.relations:
    print(f"  {names[relation.head]} -{relation.type}→ {names[relation.tail]} {relation.confidence:.2f}")

feasible: True
  e1 JeekLee (person) 1.00
  e2 무신사 (organization) 1.00
  e3 성수동 (location) 1.00
  JeekLee -lives_in→ 성수동 0.75
  JeekLee -works_for→ 무신사 0.89


| 종류 | 결과 |
|---|---|
| 엔티티 | e1 `JeekLee`(person) 1.00, e2 `무신사`(organization) 1.00, e3 `성수동`(location) 1.00 |
| 관계 | `JeekLee -lives_in→ 성수동` 0.75, `JeekLee -works_for→ 무신사` 0.89 |
| `feasible` | True |

JointIE는 관계의 끝점 타입을 스키마로 강제하므로, 예를 들어 `works_for`의 꼬리로 location이 연결되는 결과는 나오지 않는다. `unique_head=True`는 한 사람이 `works_for`를 하나만 갖도록 제한한다. 이 프로젝트의 [`GlinerExtractor`](../config/extractors.py)는 POLE+O 온톨로지 전체를 같은 방식으로 JointIE 스키마에 등록한다.

## 정리

| 목적 | 사용할 API | 참고 |
|---|---|---|
| 엔티티만 빠르게 | `extract_entities()`, `batch_extract_entities()` | 한국어는 `multi` + 형태소 분할기 |
| 문장 레이블 | `classify_text()` | |
| 필드가 정해진 레코드 | `extract_json()` | 숫자·단위가 붙은 값은 분할기에 따라 결과가 달라진다. |
| 타입 제약이 있는 그래프 | `JointIEEngine` | 엔티티 타입과 관계 제약을 스키마로 선언 |
| 여러 작업을 한 번에 | `create_schema()` + `extract()` | |

분할기 선택은 저장되지 않으므로, 모델을 불러올 때마다 `set_word_splitter()`를 다시 호출해야 한다. 한국어 데이터로 모델 자체를 개선하는 방법은 [한국어 미세조정](02_korean_fine_tuning.ipynb)에서 다룬다.